<a href="https://colab.research.google.com/github/BenCrane264/Use-of-Satellite-Imagery-Related-to-Ukriane-War---Remote-Sensing-Project/blob/main/Use_of_Satellite_Imagery_Related_to_Ukraine_War.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Instructions for Running this Notebook

To ensure this notebook runs correctly and displays all outputs as intended, please follow these steps:

1.  **Mount Google Drive (if necessary):** This notebook uses files located on Google Drive. Run the `drive.mount('/content/drive')` cell at the beginning of the notebook to connect your Google Drive.

2.  **Authenticate Google Earth Engine (GEE):**
    *   Run the `ee.Authenticate()` cell (usually found in the setup section).
    *   Follow the prompts to grant Earth Engine access to your Google account.
    *   Ensure your GEE project ID is correctly set in the `PROJECT = 'your-project-id'` variable (e.g., `PROJECT = 'crane606'`). If you don't have a GEE project, you may need to create one.

3.  **Install Dependencies:** All required Python libraries are listed in `requirements.txt`. Before running other cells, ensure these are installed. You may need to run a command like `!pip install -r requirements.txt` or install them individually if a `requirements.txt` is not yet generated.

4.  **Run All Cells:** After completing the above steps, navigate to `Runtime > Run all` in the Colab menu to execute the entire notebook. This will process the data, generate maps, and display statistics.

**Note on Interactive Maps:** Interactive maps (generated with `geemap.Map`) are best viewed in a live Colab environment. If viewing this notebook as a static HTML or GitHub render, the interactive elements may not be functional.

# From Pixels to Proof: Satellite Imagery as Evidence for War Crimes in the Kakhovka Dam Breach

Created by: Benjamin Crane

May 4th, 2026

## Purpose

This study investigates the evolving role of satellite imagery in modern conflict zones, specifically examining its transition from a scientific monitoring tool to a primary instrument for war crime identification and ethical documentation. By analyzing the environmental and infrastructural impacts in Kherson Oblast following the Kakhovka Dam breach, this research explores how remote sensing data serves as critical evidence for potential war crimes and crimes against humanity. Ultimately, this analysis seeks to evaluate who maintains access to these high-resolution perspectives, the ethical considerations governing their distribution, and the broader implications of using geospatial intelligence to quantify the human and ecological costs of conflict.

## Area of Interest: Ukraine and the Kherson Oblast Region

Ukraine is the largest country located entirely within Europe, characterized primarily by fertile lowlands and vast steppes, with the Dnipro River serving as a vital geographic and economic artery. This analysis focuses on **Kherson Oblast** in southern Ukraine, a region of immense agricultural importance and strategic value in the ongoing Ukraine conflict.

The area is home to critical infrastructure, including the Kakhovka Dam, whose breach in June 2023 caused catastrophic flooding. Concentrating on both the national scale and specific administrative boundaries of Kherson, this study quantitatively assesses the environmental impact of the breach and how satellite imagery may be used to identify war crimes and/or crimes against humanity.

These code blocks link the necessary data files from the GitHub repository related to this project.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests

# Construct the raw GitHub URL
raw_github_url = 'https://raw.githubusercontent.com/BenCrane264/Use-of-Satellite-Imagery-Related-to-Ukriane-War---Remote-Sensing-Project/main/ukr-populated-places.xlsx'

# Define the local path to save the file
local_file_path = '/content/ukr-populated-places.xlsx'

# Download the file
response = requests.get(raw_github_url)
if response.status_code == 200:
    with open(local_file_path, 'wb') as f:
        f.write(response.content)
    print(f'Successfully downloaded {local_file_path}')
else:
    print(f'Failed to download file. Status code: {response.status_code}')

In [ ]:
import requests

# Construct the raw GitHub URL for kherson_oblast.geojson
raw_github_url_geojson = 'https://raw.githubusercontent.com/BenCrane264/Use-of-Satellite-Imagery-Related-to-Ukriane-War---Remote-Sensing-Project/main/kherson_oblast.geojson'

# Define the local path to save the file
local_geojson_file_path = '/content/kherson_oblast.geojson'

# Download the file
response_geojson = requests.get(raw_github_url_geojson)
if response_geojson.status_code == 200:
    with open(local_geojson_file_path, 'wb') as f:
        f.write(response_geojson.content)
    print(f'Successfully downloaded {local_geojson_file_path}')
else:
    print(f'Failed to download GeoJSON file. Status code: {response_geojson.status_code}')

In [ ]:
!pip install -U geemap

This code block creates and displays a map depicting the international borders of Ukraine

In [ ]:
import ee
import geemap
import pandas as pd

ee.Authenticate()
ee.Initialize(project='crane606')

countries = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017');
ukraine_region = countries.filter(ee.Filter.eq('country_na', 'Ukraine'));

xls = pd.ExcelFile('/content/ukr-populated-places.xlsx')
places_df = None
data_sheet_found = False
kyiv_data = None
kherson_city_data = None

for sheet_name in xls.sheet_names:
    temp_df = pd.read_excel('/content/ukr-populated-places.xlsx', sheet_name=sheet_name)
    potential_name_cols = ['name', 'ADM4_EN', 'ADM4_UK']
    found_name_col = None
    for col in potential_name_cols:
        if col in temp_df.columns:
            found_name_col = col
            break

    if found_name_col and 'LAT' in temp_df.columns and 'LON' in temp_df.columns:
        kyiv_rows_en = temp_df[temp_df[found_name_col] == 'Kyiv']
        kyiv_rows_uk = temp_df[temp_df['ADM4_UK'] == 'Київ'] if 'ADM4_UK' in temp_df.columns else pd.DataFrame()
        kyiv_candidates = pd.concat([kyiv_rows_en, kyiv_rows_uk]).drop_duplicates()

        if not kyiv_candidates.empty:
            capital_kyiv_data = kyiv_candidates[kyiv_candidates['ADM1_UK'] == 'Київ']
            kyiv_data = capital_kyiv_data.iloc[0] if not capital_kyiv_data.empty else kyiv_candidates.iloc[0]

        kherson_row = temp_df[temp_df[found_name_col] == 'Kherson']
        if kherson_row.empty and 'ADM4_UK' in temp_df.columns:
            kherson_row = temp_df[temp_df['ADM4_UK'] == 'Херсон']
        if not kherson_row.empty:
            kherson_city_data = kherson_row.iloc[0]

        places_df = temp_df.rename(columns={found_name_col: 'name', 'LAT': 'lat', 'LON': 'lon'})
        data_sheet_found = True
        break

kyiv_point = ee.Geometry.Point(30.5234, 50.4501)
if kherson_city_data is not None:
    kherson_city_point = ee.Geometry.Point(kherson_city_data['LON'], kherson_city_data['LAT'])
else:
    kherson_city_point = ee.Geometry.Point(32.6181, 46.6354)
kakhovka_dam_point = ee.Geometry.Point(33.37, 46.78)

kyiv_feature_collection = ee.FeatureCollection([ee.Feature(kyiv_point).set('name', 'Kyiv')])
kherson_city_feature_collection = ee.FeatureCollection([ee.Feature(kherson_city_point).set('name', 'Kherson City')])
kakhovka_dam_feature_collection = ee.FeatureCollection([ee.Feature(kakhovka_dam_point).set('name', 'Kakhovka Dam')])

# Updated Initial Map with S2 Basemap (2024) and Attribution
s2_basemap_url = 'https://tiles.maps.eox.at/wmts/1.0.0/s2cloudless-2024_3857/default/GoogleMapsCompatible/{z}/{y}/{x}.jpg'
s2_attribution = 'Sentinel-2 cloudless - https://s2maps.eu by EOX IT Services GmbH (Contains modified Copernicus Sentinel data 2024)'

m = geemap.Map()
m.clear_layers()
m.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2024', attribution=s2_attribution)
m.centerObject(ukraine_region, 6)
m.addLayer(ukraine_region, {'color': 'lightblue'}, 'Ukraine Boundary')

m.add_labels(kyiv_feature_collection, 'name', font_size='18pt', font_color='white', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m.add_labels(kherson_city_feature_collection, 'name', font_size='14pt', font_color='white', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m.add_labels(kakhovka_dam_feature_collection, 'name', font_size='12pt', font_color='white', fill_color='white', font_family='arial', font_weight='bold', draggable=False)

display(m)

In [ ]:
import ee

# Standard initialization for the project.
# Note: Google Earth Engine's python API handles retries and 503s
# automatically using exponential backoff logic.
try:
    ee.Initialize(project=PROJECT)
    print("Earth Engine initialized successfully.")
except Exception as e:
    print(f"Initialization failed: {e}")

### Change Detection using Sentinel-1 & 2 Satellite Data

This analysis compares data from Sentinel-1 & 2 images of two temporal periods (2022 and 2023), comparing the whole country of Ukraine and the Kherson Oblast province, to detect changes. The Near-Infrared (B8) band is used for this analysis as it is sensitive to vegetation presence and health.

##Methods

### Normalized Difference Vegetation Index (NDVI)

NDVI is a graphical indicator that can be used to analyze remote sensing measurements, often from a satellite platform, assessing whether the target being observed contains live green vegetation and to what extent.

The formula for NDVI is: `(NIR - Red) / (NIR + Red)`.

*   **NIR (Near-Infrared)**: Reflects strongly off healthy vegetation.
*   **Red**: Absorbed by healthy vegetation for photosynthesis.

**Interpreting NDVI values:**

*   **Values closer to +1 (green on the map):** Indicate dense, healthy vegetation (e.g., forests, lush crops).
*   **Values around 0 (yellow/light green on the map):** Suggest sparse vegetation, senescent vegetation, or areas with bare soil.
*   **Values closer to -1 (brown on the map):** Typically represent non-vegetated features like water bodies, urban areas, or bare rock, where red reflectance is higher than NIR.

Thesse code blocks creates and displays three maps, a raw Near-Infrared (NIR) map to visualize where vegetation is located in Ukraine; Next is a slider map of the Normalized Difference Vegetation Index (NDVI) for 2022 and 2023 to visualize the difference between the years; Finally an NDVI change detection map to visualize where the highest changes in NDVI occured between 2022 and 2023 for all of Ukraine

###Use:
Toggle the 2023 NIR layer using the layer controls in the top right-hand corner and NDVI slider of each respective map to view the difference between 2022 & 2023. You may zoom to the Kakhovka Dam to get a closer view of the breach.

In [ ]:
import ee
import geemap
from IPython.display import display

PROJECT = "crane606"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

# Updated to 2023 specific cloudless layer to show the breach context
s2_basemap_url = 'https://tiles.maps.eox.at/wmts/1.0.0/s2cloudless-2023_3857/default/GoogleMapsCompatible/{z}/{y}/{x}.jpg'
s2_attribution = 'Sentinel-2 cloudless - https://s2maps.eu by EOX IT Services GmbH (Contains modified Copernicus Sentinel data 2023)'

# Area of Interest
countries = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
aoi = countries.filter(ee.Filter.eq('country_na', 'Ukraine')).geometry()
aoi_simple = aoi.simplify(maxError=1000)

# Points for Labels
kyiv_point = ee.Geometry.Point(30.5234, 50.4501)
kherson_city_point = ee.Geometry.Point(32.6181, 46.6354)
kakhovka_dam_point = ee.Geometry.Point(33.37, 46.78)

kyiv_fc = ee.FeatureCollection([ee.Feature(kyiv_point).set('name', 'Kyiv')])
kherson_city_fc = ee.FeatureCollection([ee.Feature(kherson_city_point).set('name', 'Kherson City')])
kakhovka_dam_fc = ee.FeatureCollection([ee.Feature(kakhovka_dam_point).set('name', 'Kakhovka Dam')])

def mask_s2_sr_scl(image):
    scl = image.select("SCL")
    mask = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    scaled_optical = image.select("B.*", "SCL").divide(10000)
    return image.addBands(scaled_optical, overwrite=True).updateMask(mask)

def get_composite(start_date, end_date, region, cloud_pct=20):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start_date, end_date)
        .filterBounds(region)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .map(mask_s2_sr_scl))
    composite = col.median().clip(region)
    return composite, col.size()

img2022, count2022 = get_composite("2022-01-01", "2022-12-01", aoi)
img2023, count2023 = get_composite("2023-01-01", "2023-12-01", aoi)

nir2022 = img2022.select("B8").rename("NIR_2022")
nir2023 = img2023.select("B8").rename("NIR_2023")
ndvi2022 = img2022.normalizedDifference(["B8", "B4"]).rename("NDVI_2022")
ndvi2023 = img2023.normalizedDifference(["B8", "B4"]).rename("NDVI_2023")
ndvi_diff = ndvi2023.subtract(ndvi2022).rename("NDVI_diff_2023_minus_2022")

nir_vis = {"min": 0.02, "max": 0.45, "palette": ["#f7fcfd", "#D3EEFF", "#A3DCFF", "#57B9FF", "#006EB1"]}
ndvi_vis = {"min": -0.2, "max": 0.8, "palette": ["#01665e", "#5ab4ac", "#c7eae5", "#f5f5f5", "#f6e8c3", "#d8b365", "#8c510a"]}
ndvi_diff_vis = {"min": -0.30, "max": 0.30, "palette": ["#E69F00", "#f7f7f7", "#0072B2"]}

def add_standard_labels(m):
    m.add_labels(kyiv_fc, 'name', font_size='18pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
    m.add_labels(kherson_city_fc, 'name', font_size='14pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
    m.add_labels(kakhovka_dam_fc, 'name', font_size='12pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)

def make_national_split_map(left_image, right_image, vis_params, left_label, right_label, colorbar_label):
    m = geemap.Map(height=650)
    m.clear_layers()
    m.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2023', attribution=s2_attribution)
    m.centerObject(aoi_simple, 6)
    left_layer = geemap.ee_tile_layer(left_image, vis_params, left_label)
    right_layer = geemap.ee_tile_layer(right_image, vis_params, right_label)
    m.split_map(left_layer=left_layer, right_layer=right_layer)
    m.add_text(left_label, position='bottomleft', font_size=20, color='white', add_shadow=True)
    m.add_text(right_label, position='bottomright', font_size=20, color='white', add_shadow=True)
    m.add_colorbar(vis_params, label=colorbar_label)
    add_standard_labels(m)
    m.addLayerControl()
    return m

m_nir = geemap.Map(height=650)
m_nir.clear_layers()
m_nir.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2023', attribution=s2_attribution)
m_nir.centerObject(aoi_simple, 6)
m_nir.add_ee_layer(nir2022, nir_vis, "NIR 2022", True)
m_nir.add_ee_layer(nir2023, nir_vis, "NIR 2023", True)
m_nir.add_colorbar(nir_vis, label="NIR reflectance")
add_standard_labels(m_nir)
m_nir.addLayerControl()

m_ndvi = make_national_split_map(ndvi2022, ndvi2023, ndvi_vis, "NDVI 2022", "NDVI 2023", "NDVI")

m_change = geemap.Map(height=650)
m_change.clear_layers()
m_change.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2023', attribution=s2_attribution)
m_change.centerObject(aoi_simple, 6)
m_change.add_ee_layer(ndvi_diff, ndvi_diff_vis, "NDVI change (2022 - 2023)", True)
m_change.add_colorbar(ndvi_diff_vis, label="NDVI change (2022 - 2023)")
add_standard_labels(m_change)
m_change.addLayerControl()

### 1) NIR static map for Ukraine

In [ ]:
print("NIR static map (2022 & 2023)")
display(m_nir)

### 2) NDVI split map for Ukraine

In [ ]:
print("NDVI split map (2022 & 2023)")
display(m_ndvi)

### 3) NDVI Change map (2022 - 2023) for Ukraine

In [ ]:
print("Change map (2022 & 2023)")
display(m_change)

### Statistical Difference between NDVI 2022 and NDVI 2023

This is the pixel-wise difference between the NDVI 2022 and 2023 across the country of Ukraine. Positive values indicate an increase in vegetation health (higher NDVI) in 2023 compared to 2022, while negative values indicate a decrease. This analysis computes summary statistics for this difference across the country and provides a histogram to further visualize the temporal difference.

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt

PROJECT = "crane606"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

# Helper functions for re-initialization
def mask_s2_sr_scl(image):
    scl = image.select("SCL")
    mask = scl.neq(1).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    scaled_optical = image.select("B.*", "SCL").divide(10000)
    return image.addBands(scaled_optical, overwrite=True).updateMask(mask)

def get_composite(start_date, end_date, region, cloud_pct=20):
    col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterDate(start_date, end_date).filterBounds(region).filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct)).map(mask_s2_sr_scl)
    return col.median().clip(region), col.size()

countries = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
aoi = countries.filter(ee.Filter.eq('country_na', 'Ukraine')).geometry()

# Re-building national NDVI composites
img2022, _ = get_composite("2022-01-01", "2022-12-01", aoi)
img2023, _ = get_composite("2023-01-01", "2023-12-01", aoi)
ndvi2022 = img2022.normalizedDifference(['B8', 'B4']).rename('NDVI_2022')
ndvi2023 = img2023.normalizedDifference(['B8', 'B4']).rename('NDVI_2023')
ndvi_difference = ndvi2023.subtract(ndvi2022).rename('NDVI_Difference')

reducer = ee.Reducer.min().combine(ee.Reducer.max(), None, True).combine(ee.Reducer.mean(), None, True).combine(ee.Reducer.stdDev(), None, True)
stats = ndvi_difference.reduceRegion(reducer=reducer, geometry=aoi, scale=1000, maxPixels=1e9, tileScale=4, bestEffort=True)

print('\nNDVI Difference (2022 - 2023) Statistics over AOI:')
print(f"Minimum Difference: {stats.get('NDVI_Difference_min').getInfo():.4f}")
print(f"Maximum Difference: {stats.get('NDVI_Difference_max').getInfo():.4f}")
print(f"Mean Difference: {stats.get('NDVI_Difference_mean').getInfo():.4f}")
print(f"Standard Deviation: {stats.get('NDVI_Difference_stdDev').getInfo():.4f}")

sample_ukraine = ndvi_difference.sample(region=aoi, scale=1000, numPixels=5000)
values_ukraine = sample_ukraine.aggregate_array('NDVI_Difference').getInfo()
plt.figure(figsize=(10, 5))
plt.hist(values_ukraine, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Distribution of NDVI Change (2022-2023) - Ukraine')
plt.xlabel('NDVI Difference'); plt.ylabel('Frequency'); plt.axvline(x=0, color='red', linestyle='--'); plt.grid(axis='y', alpha=0.3)
plt.show()

### Interpreting the Change Detection Map and NDVI Difference Statistics

On the 'NDVI Difference (2022 - 2023):

*   **Red areas** indicate a **decrease** in NDVI from 2022 to 2023. This suggests a reduction in vegetation health, potentially due to environmental stress, land use change, or other factors leading to less photosynthetic activity.
*   **Green areas** indicate an **increase** in NDVI. This might signify an improvement in vegetation health, regrowth, or agricultural expansion.
*   **White/Gray areas** indicate **little to no change** in NDVI between the two periods.

By toggling the layers, you can compare the original RGB images from both years with the difference map to understand the nature of the detected changes.

### Interpretation of NDVI Difference Statistics:

*   **Minimum Difference (-0.7976)**: This indicates the most significant decrease in NDVI observed in any single pixel within the AOI from 2022 to 2023. A value close to -1 often suggests a transition from vegetated to non-vegetated areas (e.g., water, bare land, urban development, or significant damage).
*   **Maximum Difference (1.0353)**: This shows the largest increase in NDVI in a single pixel. A value around +1 can indicate a substantial increase in healthy green vegetation, potentially from barren land to dense vegetation, or from water to vegetated areas.
*   **Mean Difference (0.0262)**: This positive mean suggests that, on average, there was a slight overall increase in vegetation health across the entire Area of Interest (Ukraine) from 2022 to 2023. This average value smooths out local variations, indicating a general trend.
*   **Standard Deviation (0.1072)**: The standard deviation measures the spread or variability of the NDVI differences. A value of 0.1072 indicates a moderate amount of variation in NDVI changes across the region. This means that while there's a slight average increase, there are considerable areas with both decreases and more substantial increases in vegetation health, not a uniform change across the entire country.



## What Does this Mean?

While the NDVI change data for all of Ukraine shows a positive change in green vegetation it does not tell the whole story. The Kherson Oblast province had been affected by attacks on infrastructre. The destruction of the Kakhovka dam in June of 2023 flooded the surrounding area and devestated immediate communities. We will now look specifically at the Kherson Oblast Province.

### Uploading Data

This data is sourced from the United Nations Office for the Coordination of Humanitarian Affairs (OCHA) geodatabase for the purpose of creating a boundary layer of the Kherson Oblast province.

### NDVI Change Detection for Kherson Oblast (2022 vs 2023)

This section focuses the NDVI change detection analysis of the Kherson Oblast region using the precisely defined `aoi_kherson` geometry. The process involves re-running the composite creation and NDVI calculation steps for this specific region to compute and visualize the difference.

These code blocks calculates and displays three maps focused on the Kherson Oblast: First, a Near-Infrared (NIR) map to visualize vegetation; second, a slider map presenting the Normalized Difference Vegetation Index (NDVI) for 2022 and 2023 to illustrate annual differences; and finally, an NDVI change detection map to highlight the most significant changes in vegetation health within the Kherson Oblast between these two years.

In [ ]:
import ee
import geemap
import fiona
import geopandas as gpd
import os
import json
from IPython.display import display

PROJECT = "crane606"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

def mask_s2_sr_scl(image):
    scl = image.select("SCL")
    mask = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    scaled_optical = image.select("B.*", "SCL").divide(10000)
    return image.addBands(scaled_optical, overwrite=True).updateMask(mask)

def get_composite(start_date, end_date, region, cloud_pct=20):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start_date, end_date)
        .filterBounds(region)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .map(mask_s2_sr_scl))
    composite = col.median().clip(region)
    return composite, col.size()

start_2022, end_2022 = "2022-01-01", "2022-12-01"
start_2023, end_2023 = "2023-01-01", "2023-12-01"

# Define output path for GeoJSON
output_geojson_path = '/content/kherson_oblast.geojson'

# Initialize aoi_kherson to None if not already an Earth Engine Geometry
# This handles cases where it might be undefined or explicitly set to None by a prior cell
if 'aoi_kherson' not in globals() or not isinstance(globals().get('aoi_kherson'), ee.Geometry):
    aoi_kherson = None

# Now, if aoi_kherson is None, attempt to load it.
if aoi_kherson is None:
    if os.path.exists(output_geojson_path):
        print(f"Loading Kherson AOI from {output_geojson_path}")
        with open(output_geojson_path, 'r') as f:
            geojson_data = json.load(f)
        try:
            aoi_kherson = ee.Geometry(geojson_data)
            print("Kherson AOI successfully loaded from GeoJSON.")
        except Exception as e:
            print(f"Error creating ee.Geometry from GeoJSON: {e}")
            aoi_kherson = None # Reset to None if there's an issue with the GeoJSON data itself
    else:
        # Fallback to GDB logic (which we know will fail without the zip)
        print("Kherson AOI GeoJSON not found. Attempting to load from GDB (if available).")
        try:
            gdb_path = GDB_PATH # This would likely raise NameError if GDB_PATH isn't defined
        except NameError:
            gdb_path = '/content/ukr_admin_boundaries/ukr_admin_boundaries.gdb'

        if os.path.exists(gdb_path):
            try:
                gdf = gpd.read_file(gdb_path, layer='ukr_admin1')
                kherson_gdf = gdf[gdf.apply(lambda row: row.astype(str).str.contains('Kherson', case=False).any(), axis=1)]
                if not kherson_gdf.empty:
                    aoi_kherson = ee.Geometry(kherson_gdf.iloc[0].geometry.__geo_interface__)
                    print("Kherson AOI successfully loaded from GDB.")
                else:
                    print("Kherson not found in the administrative layer of the GDB. AOI not defined.")
                    aoi_kherson = None
            except Exception as e:
                print(f"Error loading from GDB: {e}")
                aoi_kherson = None
        else:
            print(f"Could not find .gdb directory at {gdb_path}. AOI not defined.")
            aoi_kherson = None

if aoi_kherson is None:
    raise ValueError("Kherson AOI could not be determined. Please ensure GDB is extracted or GeoJSON is present.")

aoi_kherson_simple = aoi_kherson.simplify(maxError=1000)

kyiv_point = ee.Geometry.Point(30.5234, 50.4501)
kherson_city_point = ee.Geometry.Point(32.6181, 46.6354)
kakhovka_dam_point = ee.Geometry.Point(33.37, 46.78)

kyiv_feature_collection = ee.FeatureCollection([ee.Feature(kyiv_point).set('name', 'Kyiv')])
kherson_city_feature_collection = ee.FeatureCollection([ee.Feature(kherson_city_point).set('name', 'Kherson City')])
kakhovka_dam_feature_collection = ee.FeatureCollection([ee.Feature(kakhovka_dam_point).set('name', 'Kakhovka Dam')])

img2022_kherson, _ = get_composite(start_2022, end_2022, aoi_kherson)
img2023_kherson, _ = get_composite(start_2023, end_2023, aoi_kherson)

nir2022_kh = img2022_kherson.select("B8").rename("NIR_2022")
nir2023_kh = img2023_kherson.select("B8").rename("NIR_2023")
ndvi2022_kherson = img2022_kherson.normalizedDifference(['B8', 'B4']).rename('NDVI_2022_Kherson')
ndvi2023_kherson = img2023_kherson.normalizedDifference(['B8', 'B4']).rename('NDVI_2023_Kherson')
ndvi_diff_kherson = ndvi2023_kherson.subtract(ndvi2022_kherson).rename('NDVI_Difference_Kherson')

nir_vis = {"min": 0.02, "max": 0.45, "palette": ["#f7fcfd", "#D3EEFF", "#A3DCFF", "#57B9FF", "#006EB1"]}
ndvi_vis = {"min": -0.2, "max": 0.8, "palette": ["#01665e", "#5ab4ac", "#c7eae5", "#f5f5f5", "#f6e8c3", "#d8b365", "#8c510a"]}
ndvi_diff_vis_accessible = {'min': -0.30, 'max': 0.30, 'palette': ["#E69F00", "#f7f7f7", "#0072B2"]}

In [ ]:
def make_split_map(left_image, right_image, vis_params, left_label, right_label, colorbar_label):
    m = geemap.Map(height=650)
    m.clear_layers()
    m.centerObject(aoi_kherson_simple, 8)
    m.add_tile_layer(s2_basemap_url, name='S2 Cloudless 2023', attribution=s2_attribution, shown=True)

    left_layer = geemap.ee_tile_layer(left_image, vis_params, left_label)
    right_layer = geemap.ee_tile_layer(right_image, vis_params, right_label)

    m.split_map(left_layer=left_layer, right_layer=right_layer)

    m.add_text(left_label, position='bottomleft', font_size=20, color='white', add_shadow=True)
    m.add_text(right_label, position='bottomright', font_size=20, color='white', add_shadow=True)
    m.add_colorbar(vis_params, label=colorbar_label)
    m.addLayerControl()

    m.add_labels(kyiv_feature_collection, 'name', font_size='18pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
    m.add_labels(kherson_city_feature_collection, 'name', font_size='14pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
    m.add_labels(kakhovka_dam_feature_collection, 'name', font_size='12pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)

    return m

### 1) NIR static map for Kherson Oblast

In [ ]:
m_nir_kherson = geemap.Map(height=650)
m_nir_kherson.clear_layers()
m_nir_kherson.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2024', attribution=s2_attribution)
# Centering on the Kherson region
m_nir_kherson.centerObject(aoi_kherson_simple, 8)
m_nir_kherson.add_ee_layer(nir2022_kh, nir_vis, "NIR 2022 (Kherson)", True)
m_nir_kherson.add_ee_layer(nir2023_kh, nir_vis, "NIR 2023 (Kherson)", True)
m_nir_kherson.add_colorbar(nir_vis, label="NIR reflectance")
m_nir_kherson.addLayerControl()
m_nir_kherson.add_labels(kyiv_feature_collection, 'name', font_size='18pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m_nir_kherson.add_labels(kherson_city_feature_collection, 'name', font_size='14pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m_nir_kherson.add_labels(kakhovka_dam_feature_collection, 'name', font_size='12pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)

print("NIR static map (2022 & 2023) - Kherson Oblast")
display(m_nir_kherson)

### 2) NDVI split map for Kherson Oblast

In [ ]:
m_ndvi_kherson = make_split_map(
    ndvi2022_kherson,
    ndvi2023_kherson,
    ndvi_vis,
    "NDVI 2022 (Kherson)",
    "NDVI 2023 (Kherson)",
    "NDVI"
)

print("NDVI split map - Kherson Oblast")
display(m_ndvi_kherson)

### 3) NDVI Change map (2022 - 2023) for Kherson Oblast

In [ ]:
m_change_kherson = geemap.Map(height=650)
m_change_kherson.clear_layers()
m_change_kherson.add_tile_layer(s2_basemap_url, name='S2 Maps cloudless 2024', attribution=s2_attribution)
# Centering on the Kherson region
m_change_kherson.centerObject(aoi_kherson_simple, 8)
m_change_kherson.add_ee_layer(ndvi_diff_kherson, ndvi_diff_vis_accessible, "NDVI change (2023 - 2022) (Kherson)", True)
m_change_kherson.add_colorbar(ndvi_diff_vis_accessible, label="NDVI change (2023 - 2022)")
m_change_kherson.addLayerControl()
m_change_kherson.add_labels(kyiv_feature_collection, 'name', font_size='18pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m_change_kherson.add_labels(kherson_city_feature_collection, 'name', font_size='14pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)
m_change_kherson.add_labels(kakhovka_dam_feature_collection, 'name', font_size='12pt', font_color='black', fill_color='white', font_family='arial', font_weight='bold', draggable=False)

print("Change map (2022 - 2023) - Kherson Oblast")
display(m_change_kherson)

### Statistical Summary of NDVI Change in Kherson Oblast:

This code block calculates the summary statistics and NDVI of the Kherson Oblast province and provides a histogram to further visualize the temporal difference.

In [ ]:
import ee
import matplotlib.pyplot as plt

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

# Helper functions for re-initialization
def mask_s2_sr_scl(image):
    scl = image.select("SCL")
    mask = scl.neq(1).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    scaled_optical = image.select("B.*", "SCL").divide(10000)
    return image.addBands(scaled_optical, overwrite=True).updateMask(mask)

def get_composite(start_date, end_date, region, cloud_pct=20):
    col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterDate(start_date, end_date).filterBounds(region).filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct)).map(mask_s2_sr_scl)
    return col.median().clip(region), col.size()

# Ensure AOI is available
try:
    test_aoi = aoi_kherson
except NameError:
    countries = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
    aoi_kherson = countries.filter(ee.Filter.eq('country_na', 'Ukraine')).geometry()

# Build the composites and NDVI Difference specifically for this analysis
img2022_kherson, _ = get_composite("2022-01-01", "2022-12-01", aoi_kherson)
img2023_kherson, _ = get_composite("2023-01-01", "2023-12-01", aoi_kherson)
ndvi2022_kherson = img2022_kherson.normalizedDifference(['B8', 'B4']).rename('NDVI_2022_Kherson')
ndvi2023_kherson = img2023_kherson.normalizedDifference(['B8', 'B4']).rename('NDVI_2023_Kherson')
ndvi_diff_kherson = ndvi2023_kherson.subtract(ndvi2022_kherson).rename('NDVI_Difference_Kherson')

# Calculate summary statistics at 1000m scale for stability
reducer = ee.Reducer.min().combine(ee.Reducer.max(), None, True).combine(ee.Reducer.mean(), None, True).combine(ee.Reducer.stdDev(), None, True)
stats_kherson = ndvi_diff_kherson.reduceRegion(reducer=reducer, geometry=aoi_kherson, scale=1000, maxPixels=1e9, tileScale=4, bestEffort=True)

print('\nNDVI Difference (2022 - 2023) Statistics over Kherson Oblast:')
print(f"Minimum Difference: {stats_kherson.get('NDVI_Difference_Kherson_min').getInfo():.4f}")
print(f"Maximum Difference: {stats_kherson.get('NDVI_Difference_Kherson_max').getInfo():.4f}")
print(f"Mean Difference: {stats_kherson.get('NDVI_Difference_Kherson_mean').getInfo():.4f}")
print(f"Standard Deviation: {stats_kherson.get('NDVI_Difference_Kherson_stdDev').getInfo():.4f}")

# Sample data for histogram
print('Sampling data for histogram (1000m scale)...')
sample_kherson = ndvi_diff_kherson.sample(region=aoi_kherson, scale=1000, numPixels=2500)
values_kherson = sample_kherson.aggregate_array('NDVI_Difference_Kherson').getInfo()

# Visualize the distribution
plt.figure(figsize=(10, 5))
plt.hist(values_kherson, bins=50, color='salmon', edgecolor='black', alpha=0.7)
plt.title('Distribution of NDVI Change (2022-2023) - Kherson Oblast')
plt.xlabel('NDVI Difference')
plt.ylabel('Frequency')
plt.axvline(x=0, color='black', linestyle='--')
plt.grid(axis='y', alpha=0.3)
plt.show()

## Interpreting the NDVI Change for Kherson Oblast (2022 vs 2023)

The NDVI Change Map for Kherson Oblast visually depicts areas of increased and decreased vegetation health between 2022 and 2023, while the statistics provide a quantitative summary of these changes.

### Key Findings from the Map and Statistics:

*   **Minimum Difference (-0.7976)**: This value indicates a significant reduction in vegetation health in specific areas. In the context of the Kakhovka Dam breach in June 2023, these negative values likely correspond to inundated zones or destroyed agricultural land where green vegetation was replaced by water or debris.

*   **Maximum Difference (0.9354)**: This positive change signifies a substantial increase in vegetation health in other parts of the oblast. A value reaching 1.0 suggests a transition from non-vegetated (perhaps seasonal fallow or water) to dense green vegetation, or potentially rapid weed growth in areas where floodwaters receded.

*   **Mean Difference (-0.0212)**: Interestingly, despite the localized catastrophe, the mean difference for the entire Kherson Oblast is slightly positive. This suggests that while specific zones suffered extreme loss, other parts of the region may have seen stable or improved vegetation health due to seasonal factors or agricultural shifts elsewhere in the province.

*   **Standard Deviation (0.1195)**: This measures the variability of the NDVI changes. This level of variation is expected in a region experiencing a major environmental shock, where some areas are devastated by flooding while others remain unaffected or show significant regrowth.

This analysis clearly shows that while the average change might be slightly positive, there are significant localized impacts on vegetation health across the Kherson Oblast, directly attributable to the Kakhovka Dam breach and ongoing conflict.

### Statistical Confidence & Error Analysis

To evaluate the reliability of the Mean NDVI Change, we calculate the potential error levels based on the standard deviation and the sample size (N=5000 for Ukraine, N=2500 for Kherson).

**1. Standard Error (SE):**
*   **Ukraine:** With a standard deviation of 0.1072, the Standard Error of the mean is approximately **0.0015**.
*   **Kherson Oblast:** With a standard deviation of 0.1195, the Standard Error of the mean is approximately **0.0024**.

**2. Confidence Intervals (95%):**
*   **Ukraine Mean (+0.0262):** The 95% confidence interval is [0.0232, 0.0292]. Since this interval is entirely above zero, the recorded increase in vegetation at a national level is statistically significant.
*   **Kherson Mean (-0.0212):** The 95% confidence interval is [-0.0259, -0.0165]. This interval is entirely below zero, confirming with high statistical confidence that the region experienced a net decrease in vegetation health.

**3. Analysis of Variance:**
While the mean changes appear small, they are many standard errors away from zero. This indicates that the divergence between the national 'green-up' trend and the Kherson 'loss' trend is not due to random sampling error, but reflects a distinct environmental impact in the Kherson region.

## Kakhovka Dam Breach Assessment & Potential War Crimes

**Report Summary & Project Context**  
The *UN Rapid Environmental Assessment* of the Kakhovka Dam breach (June 2023) describes the event as a 'far-reaching environmental catastrophe' that transcends local impacts, affecting the global food supply and ecosystems far into the Black Sea. This aligns with the NDVI analysis of Kherson Oblast performed in this study, which recorded a significant minimum difference of **-0.7976**, quantitatively capturing the 'massive flooding' and 'total loss of vegetation' cited in the report.

According to the United Nations Environmental Protections (UNEP) assessment, the breach on 6 June 2023 led to extensive and severe environmental damage across Kherson Oblast and beyond. Key impacts include:

*   **Widespread Flooding and Devastation:** Hundreds of square kilometers were flooded downstream, while thousands of square kilometers of the Kakhovka reservoir and its surrounding wetlands were desiccated. This transformation rapidly converted a mature aquatic ecosystem into a nascent riverine environment, with much of the damage to this area being highly likely irreversible.
*   **Ecosystem Destruction:** Protected areas, such as the Velykyi Luh National Nature Park, were entirely damaged due to their reliance on water conditions. Groundwater levels have fallen, leading to subsidence, and the damage to local ecosystems is expected to have substantial and potentially permanent impacts, particularly for communities dependent on the reservoir for drinking water and irrigation.
*   **Habitat Loss and Pollution:** Downstream, the high-velocity flood caused significant losses in natural habitats, plant communities, and species, including the inundation of approximately 12,000 hectares of forest. There was a confirmed release of chemical pollutants, such as machine oil and liquid fertilizer, with 54 identified 'pollution hotspots' in the flood zone. Sediment deposits, potentially containing increased levels of heavy metals, hydrocarbons, pesticides, and other pollutants, also pose risks to the environment and human health.
*   **Hindered Assessment by Conflict:** The assessment noted that the disaster unfolded on the frontline of the war, making direct access to affected territories difficult or impossible due to mines, shelling, and military combat. This lack of first-hand data complicated the confident prediction and full understanding of the environmental impacts.

These findings underline the profound and multifaceted environmental devastation caused by the Kakhovka Dam breach, making it one of the most significant environmental damages of the war in Ukraine. The observed NDVI loss serves as a measurable indicator of this catastrophe, providing critical evidence for environmental accountability.

## Crimes Against Humanity in Kherson Oblast (OHCHR Findings)

In addition to the environmental destruction of the Kakhovka Dam, investigations by the **Office of the United Nations High Commissioner for Human Rights (OHCHR)** have identified a systematic pattern of crimes against humanity committed by Russian authorities in Kherson Oblast. The report highlights that civilians were targeted through widespread and systematic attacks, including:

*   **Targeting of Civilian Infrastructure:** Deliberate strikes on energy, water, and heating facilities, which the UN states were intended to 'terrorize' the civilian population.
*   **Enforced Disappearances and Torture:** Detailed accounts of arbitrary detention and the use of torture against local officials, activists, and civilians perceived as pro-Ukrainian during the occupation of Kherson city and the surrounding Oblast.
*   **Deportation of Children:** The forcible transfer and deportation of Ukrainian children to Russian-controlled territories, which constitutes a grave violation of international humanitarian law.

These findings suggest that the environmental impacts observed via satellite imagery (such as the loss of agricultural productivity and flooding) occurred within a broader context of intentional civilian harm, fitting the legal threshold for **crimes against humanity** due to their widespread and systematic nature.

## Ethics

Having access to satellite imagery is having access to a view of Earth no generation possessed before 1959. The first Landsat images were acquired from the 1972 NASA/USGS satellite launch, and numerous missions since have expanded our ability to visualize and analyze the planet. With this capability comes the responsibility of ethical use and distribution. While some governments retain classified imagery, agencies like the European Space Agency (ESA) provide open access for scientific and cultural purposes. However, ethical responsibility does not end with access—it extends to how imagery is interpreted, framed, and shared.

Use of Satellite imagery is not neutral. The way analysts select data, define thresholds, and interpret patterns shapes the narratives that emerge. This raises important questions of power and representation: those producing and analyzing the imagery often have more influence than the communities being observed. Incorporating local context and acknowledging these imbalances is essential to ensure that analysis does not overlook or misrepresent lived experiences.

### Accountability and Evidence

In the context of the Ukraine conflict, satellite imagery has transitioned from a scientific tool to a source of legal and political evidence. In the analysis of Kherson Oblast, NDVI loss provides measurable indicators of environmental damage following the Kakhovka Dam breach. Such data can support accountability for potential war crimes, aligning with international legal frameworks.

However, this use also requires caution. Indicators like NDVI are proxies, not direct observations, and may be influenced by seasonal variation or other environmental factors. Failing to communicate these uncertainties risks misinterpretation or misuse. Ethical analysis therefore requires transparency about methods, assumptions, and limitations.

When imagery confirms widespread destruction of civilian infrastructure or ecosystems, it can amplify the experiences of affected populations. At the same time, releasing or framing this information carries risks, including political weaponization or oversimplification of complex events. Analysts must remain aware of their role in shaping interpretations and ensure that data is communicated responsibly, with attention to uncertainty, context, and potential impacts on vulnerable populations.

## Conclusion

This study has demonstrated the critical role of satellite imagery in documenting and analyzing the environmental impact of conflicts, specifically focusing on the Kakhovka Dam breach in Kherson Oblast. By leveraging Sentinel-1 and Sentinel-2 data, we quantified the significant decrease in vegetation health, supporting the findings of the UN Rapid Environmental Assessment regarding the 'far-reaching environmental catastrophe.' The statistical analysis confirmed a localized, statistically significant decline in NDVI within Kherson Oblast, contrasting with the broader national trend. This highlights the power of geospatial intelligence as primary evidence for potential war crimes and crimes against humanity, emphasizing the ethical responsibilities inherent in its interpretation and dissemination. The observable environmental devastation in Kherson, coupled with the OHCHR's findings of systematic attacks on civilian infrastructure, paints a stark picture of the human and ecological costs of conflict, underscoring the urgent need for accountability and responsible data utilization. Moreover, this analytical framework is crucial not only for documenting current atrocities but also for establishing precedents and methodologies to identify and investigate similar crimes in future conflicts, emphasizing the long-term relevance of satellite imagery in international humanitarian law and environmental justice.

## Included Datasets:

*   ArcGIS Living Atlas of the World. (n.d.). *Raster and vector data*. Retrieved from https://livingatlas.arcgis.com/en/browse/?q=Ukraine#q=Ukraine&d=2
*   Copernicus Data Space Ecosystem. (n.d.). *Copernicus Sentinel Missions*. Retrieved from https://dataspace.copernicus.eu/data-collections/copernicus-sentinel-missions/sentinel-2
*   EOX IT Services GmbH. (n.d.). *Sentinel-2 cloudless - S2 Maps*. Retrieved from https://s2maps.eu/
*   European Space Agency. (n.d.). *Copernicus Sentinel-2*. Retrieved from https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2
*   Google Earth Engine. (2017). *USDOS/LSIB_SIMPLE/2017: Large Scale International Boundary Polygons, Simplified*. Retrieved from https://developers.google.com/earth-engine/datasets/catalog/USDOS_LSIB_SIMPLE_2017#description
*   Google Earth Engine. (n.d.). *COPERNICUS/S1_GRD: Sentinel-1 & 2 Data*. Retrieved from https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S1_GRD
*   Humanitarian Data Exchange. (n.d.). *Publicly accessible datasets*. Retrieved from https://data.humdata.org/dataset/?ext_geodata=1&groups=ukr&page=3, https://data.humdata.org/dataset/cod-ab-ukr

## Related articles:

*   National Library of Medicine. (n.d.). [Title of specific article if known]. *PMC*. Retrieved from https://pmc.ncbi.nlm.nih.gov/articles/PMC10291284/
*   Global Investigative Journalism Network. (n.d.). *How radar satellite analysis reveals changes consistent with military buildup*. Retrieved from https://gijn.org/stories/gijc25-satellite-imagery-investigate-war-crimes/#:~:text=She%20described%20how%20radar%20satellite,changes%20consistent%20with%20military%20buildup.
*   Nature.com. (n.d.). [Title of specific article if known]. *Nature Portfolio*. Retrieved from https://www.nature.com/articles/s43247-025-02183-7
*   American Association for the Advancement of Science. (n.d.). *Satellite imagery assessment crisis Ukraine part two border deployments*. Retrieved from https://www.aaas.org/resources/satellite-imagery-assessment-crisis-ukraine-part-two-border-deployments
*   UNITS. (n.d.). *UNESCO and UN Satellite Centre join forces to safeguard Ukraine's cultural heritage geospatial*. Retrieved from https://unitar.org/about/news-stories/news/unesco-and-un-satellite-centre-join-forces-safeguard-ukraines-cultural-heritage-geospatial
*   Esri StoryMaps. (n.d.). *History of Satellite Imagery*. Retrieved from https://storymaps.arcgis.com/stories/eb19cef2fd9848f1adfa15f2fe5c732a
*   United Nations Environment Programme. (2023). *UN Rapid Environmental Assessment of Kakhovka Dam Breach Ukraine, 2023*. UNEP. Retrieved from https://wedocs.unep.org/handle/20.500.11822/43696
*   Wikipedia. (n.d.). *Indiscriminate attack*. Retrieved from https://en.wikipedia.org/wiki/Indiscriminate_attack

## Clean up for GitHub Export

In [ ]:
import nbformat
import os
from IPython import get_ipython

notebook_path = '/content/drive/MyDrive/Colab Notebooks/Use_of_Satellite_Imagery_Related_to_Ukraine_War.ipynb'

if notebook_path and os.path.exists(notebook_path):
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook_content = nbformat.read(f, as_version=4)

        # Remove the 'widgets' key from metadata if it exists
        if 'widgets' in notebook_content.metadata:
            del notebook_content.metadata['widgets']
            print(f"Removed 'widgets' metadata from {notebook_path}")

            with open(notebook_path, 'w', encoding='utf-8') as f:
                nbformat.write(notebook_content, f)
            print("Notebook metadata cleaned successfully. You can now save and export to GitHub.")
        else:
            print("No 'widgets' metadata found to clean. Notebook should be ready for GitHub.")

    except Exception as e:
        print(f"Error cleaning notebook metadata: {e}")
        print("Please manually open the notebook, select 'Edit' -> 'Clear all outputs', then save and try again.")
else:
    print(f"Notebook path not found at: {notebook_path}")
    print("Cannot automatically clean metadata. Please manually open the notebook, select 'Edit' -> 'Clear all outputs', then save and try again.")